# SIH Member 5 — Blockchain & Audit Trail
**Self-contained Google Colab notebook • Python + SQLite + Solidity/EVM**

Version 0.2.0, reviewed against the uploaded SIH_26-main.zip. Your audit code, blockchain connector, smart contract, contract validators, tests and backend handoff are included. New risk output requires six contributions (four 0-20; barcode/watermark 0-10). You do **not** need to upload the earlier ZIP.

**First run:** select a CPU runtime and run the cells in order, or use **Runtime → Run all**. Default execution uses synthetic data and a disposable local EVM. Optional uploads, external-chain connections and downloads are disabled until you enable them.

**Integration boundary:** Member 4 supplies `RiskScoreOutput` and authenticated context → your module stores the audit event → a separate worker anchors it → Member 6 receives the verified receipt. This does not train or modify OCR, face or tampering models.

A local EVM executes real contract bytecode but is not a distributed network. Colab runtime files and the in-memory chain are temporary. Saving this notebook saves its code, not a persistent blockchain. Use the export cell to hand off source code; use a durable backend/network for shared operation.

**Reruns:** module-writing cells overwrite their named source files with the code shown. Edit those cells to keep your changes reproducible, then restart the runtime and rerun. The demo-session cell keeps an existing session in the same kernel; a new kernel gets a new journal and chain together. Never reuse an old journal with a reset chain and describe that as historical proof.


### Cell 1 — Prepare the workspace

In [ ]:
from pathlib import Path
import os, sys, json, uuid, base64, io, zipfile, subprocess
base = Path('/content') if Path('/content').exists() else Path.cwd()
REPO_WORKSPACE = base / 'sih_member5_reviewed'
PROJECT = REPO_WORKSPACE / 'modules' / 'blockchain'
PROJECT.mkdir(parents=True, exist_ok=True)
for folder in ('member5', 'contracts', 'examples', 'tests', 'scripts'):
    (PROJECT / folder).mkdir(exist_ok=True)
os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
print('Workspace:', PROJECT)
print('No GPU is required for Member 5 code.')

The next cell restores the bundled test files, compiled contract artifact, examples and handoff documents. The main modules follow as readable, editable code cells. No remote repository is downloaded.

### Cell 2 — Restore supporting files

In [ ]:
# @title Restore supporting files
SUPPORT_ZIP_B64 = '...'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(SUPPORT_ZIP_B64))) as archive:
    for name in archive.namelist():
        destination = (PROJECT / name).resolve()
        if PROJECT.resolve() not in destination.parents:
            raise ValueError('Invalid embedded archive path')
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(archive.read(name))
print('Supporting files ready; reviewed contracts are in contract_reference/.')

### Cell 3 — Install notebook dependencies

In [ ]:
# This switch is for local automated verification only; leave it unset in Colab.
if os.environ.get('SIH_NOTEBOOK_TEST_SKIP_INSTALL') != '1':
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet',
                           'web3==7.16.0', 'eth-tester[py-evm]==0.14.0b1',
                           'py-solc-x==2.0.5'])
print('Dependencies ready. Core audit code itself uses only the standard library.')

## Separate source modules
These cells create importable `.py` files. Keep these modules separate when handing them to Member 4. The Solidity source and compiled artifact must match; after a Solidity edit, rebuild using `python scripts/compile_contract.py` before deploying. Python module edits require restarting imports/the runtime before they take effect.

### Cell 4 — member5/api_adapter.py

In [ ]:
%%writefile member5/api_adapter.py
"""Optional in-process helpers for contracts/common.ts ApiResponse<T>.

No HTTP server is created. Exceptions propagate for Member 4's error handler.
The shared contract requires data:T even on failure, so we do NOT silently
invent a null error shape or return a fake successful audit result.
"""
from .audit_trail import text_id
from .integration import record_verification, get_audit_output


def success_response(data, request_id):
    text_id(request_id)
    return {'success': True, 'data': data, 'requestId': request_id}


def record_verification_response(audit, payload, *, request_id, **context):
    text_id(request_id)  # Validate before the side effect.
    return success_response(record_verification(audit, payload, **context), request_id)


def record_risk_response(audit, risk_response, *, document_id, checkpoint_id,
                         timestamp, **context):
    """Accept ApiResponse<RiskScoreOutput> from Member 4 and unwrap data.

    The upstream requestId is preserved as a correlation ID. It is NOT the
    idempotency key: context still requires stable event_id/verification_id.
    """
    required = {'success', 'data', 'requestId'}
    if (not isinstance(risk_response, dict) or not required <= set(risk_response)
            or set(risk_response) - required - {'error'}):
        raise ValueError('Expected ApiResponse<RiskScoreOutput>')
    if risk_response['success'] is not True or risk_response.get('error'):
        raise ValueError('Risk response failed; no final-decision audit record written')
    text_id(risk_response['requestId'])
    payload = {'documentId': document_id, 'checkpointId': checkpoint_id,
               'timestamp': timestamp, 'riskResult': risk_response['data']}
    return record_verification_response(audit, payload,
        request_id=risk_response['requestId'], **context)


def get_audit_response(audit, chain, event_id, *, request_id):
    """ApiResponse<AuditLogOutput | AuditPendingAck>. Errors propagate."""
    text_id(request_id)
    confirmed = get_audit_output(audit, chain, event_id)
    if confirmed is not None:
        return success_response(confirmed, request_id)
    event = audit.get_event(event_id)
    pending = {'eventId': event_id, 'sequence': event['sequence'],
               'eventHash': event['event_hash'], 'auditStored': True,
               'status': 'pending', 'logged': False, 'warnings': []}
    return success_response(pending, request_id)

### Cell 5 — member5/audit_trail.py

In [ ]:
%%writefile member5/audit_trail.py
"""Off-chain, append-only-by-API audit journal. No blockchain dependency."""
import hashlib
import json
import secrets
import sqlite3
from contextlib import closing
from datetime import datetime, timezone

ZERO = '0' * 64


def canonical(value):
    """Protocol v1: Python JSON, sorted keys, compact separators, ASCII escaping.

    Store these exact bytes; this is NOT RFC 8785. Cross-language verifiers must
    hash the stored UTF-8 string, not deserialize and reserialize it.
    """
    return json.dumps(value, sort_keys=True, separators=(',', ':'),
                      ensure_ascii=True, allow_nan=False)


def sha256(data):
    return hashlib.sha256(data).hexdigest()


def digest_hex(value):
    if not isinstance(value, str) or len(value) != 64:
        raise ValueError('Expected lowercase SHA-256 hex')
    if any(c not in '0123456789abcdef' for c in value):
        raise ValueError('Expected lowercase SHA-256 hex')
    return value


def text_id(value):
    if not isinstance(value, str) or not 1 <= len(value) <= 160:
        raise ValueError('IDs must be strings of 1..160 characters')
    return value


class AuditTrail:
    def __init__(self, path):
        self.path = str(path)
        with closing(self.connect()) as db:
            db.executescript('''
            PRAGMA journal_mode=WAL;
            CREATE TABLE IF NOT EXISTS events (
                sequence INTEGER PRIMARY KEY,
                event_id TEXT UNIQUE NOT NULL,
                request_json TEXT NOT NULL,
                envelope_json TEXT NOT NULL,
                event_hash TEXT NOT NULL);
            CREATE TRIGGER IF NOT EXISTS no_update BEFORE UPDATE ON events
            BEGIN SELECT RAISE(ABORT, 'Audit events cannot be updated'); END;
            CREATE TRIGGER IF NOT EXISTS no_delete BEFORE DELETE ON events
            BEGIN SELECT RAISE(ABORT, 'Audit events cannot be deleted'); END;
            ''')

    def connect(self):
        db = sqlite3.connect(self.path, timeout=30, isolation_level=None)
        db.row_factory = sqlite3.Row
        return db

    def append(self, *, event_id, verification_id, actor_id, event_type,
               document_sha256, result):
        """Trusted backend only. Same ID + same payload returns original event.

        result must be JSON-compatible; sanitize it before calling. Exceptions
        mean no acknowledgement: caller should retry using the SAME event_id.
        """
        for value in (event_id, verification_id, actor_id, event_type):
            text_id(value)
        digest_hex(document_sha256)
        if not isinstance(result, dict):
            raise ValueError('result must be an object')
        request = canonical(dict(event_id=event_id, verification_id=verification_id,
                                 actor_id=actor_id, event_type=event_type,
                                 document_sha256=document_sha256, result=result))
        if len(request.encode()) > 262144:
            raise ValueError('Event exceeds 256 KiB; store artifacts separately')
        db = self.connect()
        try:
            db.execute('BEGIN IMMEDIATE')
            old = db.execute('SELECT * FROM events WHERE event_id=?', (event_id,)).fetchone()
            if old:
                if old['request_json'] != request:
                    raise ValueError('event_id already used for a different payload')
                db.commit()
                return dict(old)
            last = db.execute('SELECT sequence,event_hash FROM events ORDER BY sequence DESC LIMIT 1').fetchone()
            seq, previous = (last['sequence'] + 1, last['event_hash']) if last else (1, ZERO)
            envelope = canonical(dict(schema='member5.audit.v1', sequence=seq,
                previous_hash=previous, recorded_at=datetime.now(timezone.utc).isoformat(),
                salt=secrets.token_hex(32), request=json.loads(request)))
            event_hash = sha256(envelope.encode())
            db.execute('INSERT INTO events VALUES (?,?,?,?,?)',
                       (seq, event_id, request, envelope, event_hash))
            db.commit()
            return dict(sequence=seq, event_id=event_id, request_json=request,
                        envelope_json=envelope, event_hash=event_hash)
        except Exception:
            db.rollback()
            raise
        finally:
            db.close()

    def events(self):
        with closing(self.connect()) as db:
            return [dict(row) for row in db.execute('SELECT * FROM events ORDER BY sequence')]

    def get_event(self, event_id):
        with closing(self.connect()) as db:
            row = db.execute('SELECT * FROM events WHERE event_id=?', (event_id,)).fetchone()
            if row is None:
                raise KeyError(event_id)
            return dict(row)

    def history(self, verification_id):
        """Trusted backend only; authorize the requester before returning details."""
        return [row for row in self.events() if
                json.loads(row['envelope_json'])['request']['verification_id'] == verification_id]

    def verify_local(self):
        """Internal consistency only: does NOT prove anchoring or completeness."""
        previous = ZERO
        rows = self.events()
        for seq, row in enumerate(rows, 1):
            body = json.loads(row['envelope_json'])
            if (row['sequence'] != seq or body['sequence'] != seq or
                body['previous_hash'] != previous or
                body['request']['event_id'] != row['event_id'] or
                canonical(body['request']) != row['request_json'] or
                sha256(row['envelope_json'].encode()) != row['event_hash']):
                raise ValueError(f'Audit corruption at sequence {seq}')
            previous = row['event_hash']
        return {'count': len(rows), 'head': previous, 'local_consistency': True}

### Cell 6 — member5/contracts.py

In [ ]:
%%writefile member5/contracts.py
"""Input validation aligned with the uploaded SIH_26 repository snapshot.

Six output contributions are REQUIRED, even when barcode/watermark inputs to
Member 4's risk engine were optional. We never invent contributions for them.
"""
import math
from datetime import datetime
from .audit_trail import text_id

BREAKDOWN_LIMITS = {
    'ocr': 20, 'validation': 20, 'tampering': 20, 'face': 20,
    'barcode': 10, 'watermark': 10,
}


def exact_keys(obj, names, label):
    if not isinstance(obj, dict) or set(obj) != set(names):
        raise ValueError(f'{label} requires exactly: {", ".join(names)}')


def number(value, maximum, label):
    # Range-check before isfinite: huge invalid JSON integers need not overflow
    # a float conversion. bool is not accepted as a numeric contribution.
    if (type(value) not in (int, float) or not 0 <= value <= maximum
            or not math.isfinite(value)):
        raise ValueError(f'{label} must be finite and between 0 and {maximum}')


def validate_input(payload):
    exact_keys(payload, ['documentId', 'checkpointId', 'timestamp', 'riskResult'], 'AuditLogInput')
    text_id(payload['documentId'])
    text_id(payload['checkpointId'])
    timestamp = payload['timestamp']
    if not isinstance(timestamp, str) or 'T' not in timestamp:
        raise ValueError('timestamp must be ISO 8601 with time and timezone')
    try:
        parsed = datetime.fromisoformat(timestamp.replace('Z', '+00:00'))
    except ValueError as exc:
        raise ValueError('Invalid ISO 8601 timestamp') from exc
    if parsed.tzinfo is None:
        raise ValueError('timestamp must include a timezone')
    risk = payload['riskResult']
    exact_keys(risk, ['score', 'decision', 'summary', 'breakdown'], 'RiskScoreOutput')
    number(risk['score'], 100, 'score')
    if risk['decision'] not in ('accept', 'flag', 'reject'):
        raise ValueError('decision must be accept, flag, or reject')
    if not isinstance(risk['summary'], str) or len(risk['summary']) > 8192:
        raise ValueError('summary must be a string of at most 8192 characters')
    exact_keys(risk['breakdown'], BREAKDOWN_LIMITS, 'RiskScoreBreakdown (repository six-field contract)')
    for key, maximum in BREAKDOWN_LIMITS.items():
        number(risk['breakdown'][key], maximum, key)
    warnings = []
    if not math.isclose(sum(risk['breakdown'].values()), risk['score'], abs_tol=0.000001):
        warnings.append('BREAKDOWN_SUM_DIFFERS_FROM_SCORE')
    return warnings

### Cell 7 — member5/integration.py

In [ ]:
%%writefile member5/integration.py
"""Member 4/6 adapter. Input field names match the supplied TypeScript contract."""
from .audit_trail import sha256, text_id
from .contracts import validate_input


def record_verification(audit, payload, *, event_id, verification_id,
                        actor_id, original_document_bytes):
    """Accept AuditLogInput. Return durable local ACK, NOT AuditLogOutput.

    event_id: persist once per logical event, reuse on retries.
    verification_id: new opaque ID per screening attempt (not per document).
    actor_id: authenticated backend identity, never browser-supplied identity.
    Original bytes are hashed; neither stored here nor sent to the chain.
    """
    if not isinstance(original_document_bytes, bytes):
        raise ValueError('Pass original upload bytes, not a decoded image')
    warnings = validate_input(payload)
    event = audit.append(event_id=event_id, verification_id=verification_id,
        actor_id=actor_id, event_type='VERIFICATION_COMPLETED',
        document_sha256=sha256(original_document_bytes), result=payload)
    return {'eventId': event_id, 'sequence': event['sequence'],
            'eventHash': event['event_hash'], 'auditStored': True,
            'status': 'pending', 'logged': False, 'warnings': warnings}


STAGES = {'DOCUMENT_UPLOADED', 'OCR_COMPLETED', 'VALIDATION_COMPLETED',
          'TAMPERING_COMPLETED', 'FACE_COMPLETED', 'PROCESSING_FAILED',
          'OFFICER_OVERRIDE', 'BARCODE_COMPLETED', 'WATERMARK_COMPLETED'}


def record_stage(audit, *, event_id, verification_id, actor_id, document_sha256,
                 stage, checkpoint_id, details):
    """Optional chronological events. Backend invokes at each actual stage.

    details: sanitized JSON including model/rule version, outcome or reason code.
    Do not include images, OCR PII, embeddings, access tokens or stack traces.
    OFFICER_OVERRIDE should reference the prior eventId and state the reason.
    """
    if stage not in STAGES:
        raise ValueError('Unknown stage')
    text_id(checkpoint_id)
    if not isinstance(details, dict):
        raise ValueError('details must be an object')
    return audit.append(event_id=event_id, verification_id=verification_id,
        actor_id=actor_id, event_type=stage, document_sha256=document_sha256,
        result={'checkpointId': checkpoint_id, 'details': details})


def get_audit_output(audit, chain, event_id):
    """Return original AuditLogOutput only for a confirmed matching anchor.

    None means not yet anchored. RPC/corruption errors propagate, never success.
    """
    from .blockchain import verify_anchoring
    status = verify_anchoring(audit, chain)
    event = audit.get_event(event_id)
    if event['sequence'] > status['anchored_count']:
        return None
    return chain.receipt_output(event['sequence'], event['event_hash'])


def verify_document(audit, event_id, document_bytes, chain=None):
    """Compare exact file bytes; this does NOT classify a document as genuine."""
    import json
    audit.verify_local()
    event = audit.get_event(event_id)
    original_hash = json.loads(event['envelope_json'])['request']['document_sha256']
    output = get_audit_output(audit, chain, event_id) if chain is not None else None
    return {'fileMatchesRecorded': sha256(document_bytes) == original_hash,
            'anchorStatus': 'not_checked' if chain is None else ('anchored' if output else 'pending'),
            'auditOutput': output}

### Cell 8 — member5/blockchain.py

In [ ]:
%%writefile member5/blockchain.py
"""Optional real EVM connector. Signing key stays in server environment."""
import json
from datetime import datetime, timezone
from .audit_trail import ZERO, digest_hex

# Minimal ABI for the exact supplied contract; Solidity artifact not needed to call it.
def function(name, inputs, outputs, mutability='view'):
    return dict(type='function', name=name, stateMutability=mutability,
                inputs=[dict(name=n, type=t) for n,t in inputs],
                outputs=[dict(name='', type=t) for t in outputs])

ABI = [function('count', [], ['uint256']),
       function('head', [], ['bytes32']),
       function('hashes', [('sequence','uint256')], ['bytes32']),
       function('writers', [('writer','address')], ['bool']),
       function('anchor', [('sequence','uint256'), ('previous','bytes32'),
                           ('digest','bytes32')], [], 'nonpayable'),
       {'type': 'event', 'name': 'Anchored', 'anonymous': False, 'inputs': [
           {'name': 'sequence', 'type': 'uint256', 'indexed': True},
           {'name': 'digest', 'type': 'bytes32', 'indexed': False},
           {'name': 'writer', 'type': 'address', 'indexed': True}]}]


class EVMAnchor:
    def __init__(self, rpc_url, contract_address, chain_id, private_key=None,
                 confirmations=1, deployment_block=0, provider=None):
        from web3 import Web3
        if confirmations < 1:
            raise ValueError('confirmations must be >= 1')
        self.w3 = Web3(provider if provider is not None else
                       Web3.HTTPProvider(rpc_url, request_kwargs={'timeout': 30}))
        if self.w3.eth.chain_id != int(chain_id):
            raise ValueError('Wrong blockchain network')
        address = Web3.to_checksum_address(contract_address)
        if not self.w3.eth.get_code(address):
            raise ValueError('No contract deployed at configured address')
        self.contract = self.w3.eth.contract(address=address, abi=ABI)
        self.account = self.w3.eth.account.from_key(private_key) if private_key else None
        self.confirmations = confirmations
        self.deployment_block = int(deployment_block)

    def receipt_output(self, sequence, expected_digest):
        count, _, block_hash = self.snapshot()
        if sequence > count or self.hash_at(sequence, block_hash) != expected_digest:
            raise ValueError('Anchor not confirmed or mismatched')
        height = self.w3.eth.get_block(block_hash)['number']
        matches = []
        # Bounded ranges support RPC providers that cap eth_getLogs spans.
        for start in range(self.deployment_block, height + 1, 1000):
            matches.extend(self.contract.events.Anchored().get_logs(
                from_block=start, to_block=min(start + 999, height),
                argument_filters={'sequence': sequence}))
        if len(matches) != 1:
            raise ValueError('Expected one original anchor event; check deployment block/RPC')
        log = matches[0]
        if bytes(log['args']['digest']).hex() != expected_digest:
            raise ValueError('Anchor event digest mismatch')
        mined_block = self.w3.eth.get_block(log['blockNumber'])
        if mined_block['hash'] != log['blockHash']:
            raise ValueError('Blockchain changed during verification; retry')
        receipt = self.w3.eth.get_transaction_receipt(log['transactionHash'])
        if receipt['status'] != 1 or receipt['blockHash'] != log['blockHash']:
            raise ValueError('Receipt missing or changed; retry')
        # Check the original snapshot is still canonical after the reads.
        if self.w3.eth.get_block(height)['hash'] != block_hash:
            raise ValueError('Blockchain reorganized; retry')
        return {'txHash': self.w3.to_hex(log['transactionHash']),
                'blockNumber': log['blockNumber'],
                'timestamp': datetime.fromtimestamp(mined_block['timestamp'], timezone.utc).isoformat(),
                'logged': True,
                'ledger': f'EVM chain {self.w3.eth.chain_id} / {self.contract.address}'}

    def snapshot(self):
        # One block for all reads; wait for configured confirmation depth.
        height = max(0, self.w3.eth.block_number - self.confirmations + 1)
        block_hash = self.w3.eth.get_block(height)['hash']
        code = self.w3.eth.get_code(self.contract.address, block_identifier=height)
        if self.w3.eth.get_block(height)['hash'] != block_hash:
            raise ValueError('Blockchain reorganized during snapshot; retry')
        if not code:
            # The contract deployment itself may still be awaiting confirmations.
            return 0, ZERO, block_hash
        count = self.contract.functions.count().call(block_identifier=block_hash)
        head = bytes(self.contract.functions.head().call(block_identifier=block_hash)).hex()
        return count, head, block_hash

    def hash_at(self, sequence, block):
        return bytes(self.contract.functions.hashes(sequence).call(block_identifier=block)).hex()

    def submit(self, sequence, previous, digest):
        if self.account is None:
            raise ValueError('Worker requires signing key')
        digest_hex(previous)
        digest_hex(digest)
        sender = self.account.address
        if self.contract.functions.count().call() >= sequence:
            current = bytes(self.contract.functions.hashes(sequence).call()).hex()
            if current != digest:
                raise ValueError('Conflicting existing anchor')
            return {'status': 'awaiting_confirmations', 'sequence': sequence}
        if not self.contract.functions.writers(sender).call():
            raise ValueError('Account is not an authorized writer')
        # Single worker / dedicated account required for nonce ownership.
        if self.w3.eth.get_transaction_count(sender, 'pending') != self.w3.eth.get_transaction_count(sender, 'latest'):
            return {'status': 'pending_transaction', 'sequence': sequence}
        tx = self.contract.functions.anchor(sequence, bytes.fromhex(previous), bytes.fromhex(digest)).build_transaction({
            'from': sender, 'nonce': self.w3.eth.get_transaction_count(sender, 'pending'),
            'chainId': self.w3.eth.chain_id})
        signed = self.account.sign_transaction(tx)
        tx_hash = self.w3.eth.send_raw_transaction(signed.raw_transaction)
        # A timeout is UNKNOWN, never success. Next run reconciles contract state.
        receipt = self.w3.eth.wait_for_transaction_receipt(tx_hash, timeout=60)
        if receipt['status'] != 1:
            raise RuntimeError('Anchor transaction reverted')
        return {'status': 'mined', 'transaction_hash': self.w3.to_hex(tx_hash),
                'sequence': sequence, 'block_number': receipt['blockNumber']}


def verify_anchoring(audit, chain):
    audit.verify_local()
    rows = audit.events()
    count, head, block = chain.snapshot()
    if count > len(rows):
        raise ValueError('Local journal is missing anchored events (truncation/restore)')
    for row in rows[:count]:
        if chain.hash_at(row['sequence'], block) != row['event_hash']:
            raise ValueError(f"Blockchain mismatch at sequence {row['sequence']}")
    expected = rows[count-1]['event_hash'] if count else ZERO
    if head != expected:
        raise ValueError('Blockchain head mismatch')
    return {'anchored_count': count, 'pending_count': len(rows)-count,
            'status': 'anchored' if count == len(rows) and count else 'pending'}


def anchor_next(audit, chain):
    """Ordered retryable outbox: events themselves are durable pending work.

    Run exactly one worker per database and dedicated signing account.
    Never call this inside a model inference HTTP request.
    """
    status = verify_anchoring(audit, chain)
    rows = audit.events()
    count = status['anchored_count']
    if count == len(rows):
        return status
    row = rows[count]
    envelope = json.loads(row['envelope_json'])
    return chain.submit(row['sequence'], envelope['previous_hash'], row['event_hash'])

### Cell 9 — member5/worker.py

In [ ]:
%%writefile member5/worker.py
"""Run periodically under ONE supervised process: python -m member5.worker."""
import json
import os
import sqlite3
from .audit_trail import AuditTrail
from .blockchain import EVMAnchor, anchor_next


def main():
    path = os.environ.get('AUDIT_DB', 'audit.sqlite3')
    # Separate SQLite lock releases automatically after crashes. It does not
    # hold the audit database write lock while waiting for the network.
    lock = sqlite3.connect(path + '.worker-lock', timeout=0, isolation_level=None)
    try:
        lock.execute('BEGIN EXCLUSIVE')
        chain = EVMAnchor(os.environ['RPC_URL'], os.environ['CONTRACT_ADDRESS'],
            int(os.environ['CHAIN_ID']), os.environ['AUDIT_PRIVATE_KEY'],
            int(os.environ.get('CONFIRMATIONS', '1')),
            int(os.environ.get('DEPLOYMENT_BLOCK', '0')))
        result = anchor_next(AuditTrail(path), chain)
        print(json.dumps(result, default=str))
    finally:
        lock.close()


if __name__ == '__main__':
    main()

### Cell 10 — member5/cli.py

In [ ]:
%%writefile member5/cli.py
"""Local trusted-operator CLI; no public HTTP endpoint or login system."""
import argparse
import json
import os
from pathlib import Path
from .audit_trail import AuditTrail
from .integration import record_verification, verify_document


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--db', default='audit.sqlite3')
    sub = parser.add_subparsers(dest='command', required=True)
    record = sub.add_parser('record')
    for flag in ('input', 'document', 'event-id', 'verification-id', 'actor-id'):
        record.add_argument('--' + flag, required=True)
    history = sub.add_parser('history')
    history.add_argument('--verification-id', required=True)
    verify = sub.add_parser('verify')
    verify.add_argument('--event-id', required=True)
    verify.add_argument('--document', required=True)
    verify.add_argument('--chain', action='store_true')
    args = parser.parse_args()
    audit = AuditTrail(args.db)
    if args.command == 'record':
        result = record_verification(audit, json.loads(Path(args.input).read_text()),
            event_id=args.event_id, verification_id=args.verification_id,
            actor_id=args.actor_id, original_document_bytes=Path(args.document).read_bytes())
    elif args.command == 'history':
        audit.verify_local()
        result = audit.history(args.verification_id)
    else:
        chain = None
        if args.chain:
            from .blockchain import EVMAnchor
            chain = EVMAnchor(os.environ['RPC_URL'], os.environ['CONTRACT_ADDRESS'],
                int(os.environ['CHAIN_ID']),
                confirmations=int(os.environ.get('CONFIRMATIONS', '1')),
                deployment_block=int(os.environ.get('DEPLOYMENT_BLOCK', '0')))
        result = verify_document(audit, args.event_id, Path(args.document).read_bytes(), chain)
    print(json.dumps(result, indent=2))

if __name__ == '__main__': main()

### Cell 11 — member5/__init__.py

In [ ]:
%%writefile member5/__init__.py
from .audit_trail import AuditTrail, sha256
from .integration import record_verification

__all__ = ['AuditTrail', 'sha256', 'record_verification']

### Cell 12 — contracts/AuditAnchor.sol

In [ ]:
%%writefile contracts/AuditAnchor.sol
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.24;

/// One deployment per journal. No PII, document hash, or risk score on-chain.
contract AuditAnchor {
    address public immutable owner;
    mapping(address => bool) public writers;
    mapping(uint256 => bytes32) public hashes;
    uint256 public count;
    bytes32 public head;

    event Anchored(uint256 indexed sequence, bytes32 digest, address indexed writer);
    event WriterChanged(address indexed writer, bool allowed);

    constructor() {
        owner = msg.sender;
        writers[msg.sender] = true;
    }

    function setWriter(address writer, bool allowed) external {
        require(msg.sender == owner, "owner only");
        require(writer != address(0), "zero writer");
        writers[writer] = allowed;
        emit WriterChanged(writer, allowed);
    }

    function anchor(uint256 sequence, bytes32 previous, bytes32 digest) external {
        require(writers[msg.sender], "writer only");
        require(digest != bytes32(0), "zero digest");
        // Exact retries are safe; rewriting any old commitment is impossible.
        if (sequence > 0 && sequence <= count) {
            require(hashes[sequence] == digest, "conflicting retry");
            return;
        }
        require(sequence == count + 1 && previous == head, "out of order");
        hashes[sequence] = digest;
        head = digest;
        count = sequence;
        emit Anchored(sequence, digest, msg.sender);
    }
}

### Cell 13 — Import your module and check the contract artifact

In [ ]:
import hashlib, copy, sqlite3, tempfile
from datetime import datetime, timezone
from web3 import Web3, EthereumTesterProvider
from member5 import AuditTrail, sha256, record_verification
from member5.api_adapter import record_risk_response, get_audit_response
from member5.integration import record_stage, get_audit_output, verify_document
from member5.blockchain import EVMAnchor, anchor_next, verify_anchoring
from member5.audit_trail import canonical
artifact = json.loads((PROJECT / 'contracts/AuditAnchor.json').read_text())
assert artifact['source_sha256'] == hashlib.sha256(
    (PROJECT / 'contracts/AuditAnchor.sol').read_bytes()).hexdigest(), 'Recompile edited Solidity source.'
print('Imports and source/bytecode provenance check passed.')

## Repository compatibility gate
The reviewed root TypeScript files are bundled under contract_reference/. This cell verifies that frozen snapshot. In your real checkout, run `python modules/blockchain/scripts/verify_contracts.py` to compare the root contracts. This notebook does not fetch live GitHub changes.

The supplied backend currently contains invalid numbered Python fragments, beginning with `2.REFERENCE_FILE`. This independent notebook does not import that backend or claim end-to-end AI integration.

### Cell 14 — Verify the reviewed repository contract snapshot

In [ ]:
subprocess.check_call([sys.executable, 'scripts/verify_contracts.py',
                       '--contracts', str(PROJECT / 'contract_reference')])
from member5.contracts import BREAKDOWN_LIMITS
print('Required contribution ranges:', BREAKDOWN_LIMITS)
print('Do not fill missing barcode/watermark contributions automatically.')

## Integration entry point
This callable accepts the **exact `AuditLogInput` object** from your PDF. The extra keyword arguments are trusted backend context, not changes to the JSON contract. Member 4 must persist the event ID and reuse it for retries. A new screening attempt needs a new verification ID.

The backend derives `actor_id` and checkpoint authorization from its login context. Supplying a name in Colab is only demonstration identity, not authentication.

### Cell 15 — Backend-ready callable

In [ ]:
def submit_screening_to_audit(audit_store, audit_log_input, original_file_bytes, *,
                              event_id, verification_id, actor_id):
    """Call after Member 4 has produced and persisted the final risk decision.

    Returns local pending ACK; the blockchain worker runs separately.
    No model inference or risk-score calculation occurs here.
    """
    return record_verification(
        audit_store, audit_log_input,
        event_id=event_id, verification_id=verification_id,
        actor_id=actor_id, original_document_bytes=original_file_bytes,
    )
print('submit_screening_to_audit(...) is ready.')

### Cell 16 — Create a disposable local EVM and matching audit journal

In [ ]:
if 'demo_chain' not in globals():
    demo_session_id = uuid.uuid4().hex
    provider = EthereumTesterProvider()
    w3 = Web3(provider)
    factory = w3.eth.contract(abi=artifact['abi'], bytecode=artifact['bin'])
    deployment_tx = factory.constructor().transact({'from': w3.eth.accounts[0]})
    deployment_receipt = w3.eth.wait_for_transaction_receipt(deployment_tx)
    # Only the in-memory tester's generated key. Never print signing keys.
    demo_key = provider.ethereum_tester.backend.account_keys[0].to_hex()
    demo_chain = EVMAnchor('', deployment_receipt['contractAddress'],
        w3.eth.chain_id, demo_key, provider=provider,
        deployment_block=deployment_receipt['blockNumber'])
    demo_audit = AuditTrail(PROJECT / f'demo-{demo_session_id}.sqlite3')
    demo_verification_id = 'screening-' + demo_session_id
    demo_timestamp = datetime.now(timezone.utc).isoformat()
    del demo_key
print('Mode: disposable in-memory EVM')
print('Contract:', demo_chain.contract.address)
print('Journal:', demo_audit.path)
print('Chain and journal are reused when this cell is rerun in the same kernel.')

### Cell 17 — Risk result input — replace this shape with Member 4 output

In [ ]:
# Synthetic example only. These are not predictions from your AI models.
original_document_bytes = b'SIH SYNTHETIC SAMPLE DOCUMENT - NOT A REAL ID'
audit_input = {
    'documentId': 'internal-demo-doc-001',
    'checkpointId': 'checkpoint-demo-A',
    'timestamp': demo_timestamp,
    'riskResult': {
        'score': 78,
        'decision': 'flag',
        'summary': 'Synthetic example: recommend manual review.',
        'breakdown': {'ocr': 18, 'validation': 16, 'tampering': 18, 'face': 16,
                      'barcode': 6, 'watermark': 4},
    },
}
print(json.dumps(audit_input, indent=2))
print('Original-file SHA-256:', sha256(original_document_bytes))

The repository risk example uses score 78 with contributions totaling 64. This example totals 78. The adapter flags a mismatch with `BREAKDOWN_SUM_DIFFERS_FROM_SCORE` but preserves Member 4’s original decision. It never invents new scoring rules.

### Cell 18 — Record an upload stage and the final decision

In [ ]:
upload_event = record_stage(
    demo_audit, event_id=demo_verification_id + ':upload',
    verification_id=demo_verification_id, actor_id='demo-officer',
    document_sha256=sha256(original_document_bytes), stage='DOCUMENT_UPLOADED',
    checkpoint_id=audit_input['checkpointId'], details={'status': 'received'},
)
completion_event_id = demo_verification_id + ':completed'
risk_api_response = {'success': True, 'data': audit_input['riskResult'],
                     'requestId': 'risk-' + demo_verification_id}
response = record_risk_response(
    demo_audit, risk_api_response, document_id=audit_input['documentId'],
    checkpoint_id=audit_input['checkpointId'], timestamp=audit_input['timestamp'],
    event_id=completion_event_id, verification_id=demo_verification_id,
    actor_id='demo-officer', original_document_bytes=original_document_bytes,
)
ack = response['data']
print(json.dumps(response, indent=2))
print('success=True means API success; data.logged=False is still pending.')
print('This ACK only confirms local storage. Query the chain for current anchor status.')

### Cell 19 — Anchor pending demo events

In [ ]:
# In the actual backend, this belongs in the separate supervised worker.
# One confirmation is used only for this immediate-mining local demonstration.
for _ in range(len(demo_audit.events())):
    progress = anchor_next(demo_audit, demo_chain)
    print(progress)
    if progress['status'] in ('pending_transaction', 'awaiting_confirmations', 'anchored'):
        break
print(json.dumps(verify_anchoring(demo_audit, demo_chain), indent=2))

### Cell 20 — Read the exact AuditLogOutput for Member 6

In [ ]:
confirmed_output = get_audit_output(demo_audit, demo_chain, completion_event_id)
assert confirmed_output is not None
assert set(confirmed_output) == {'txHash', 'blockNumber', 'timestamp', 'logged', 'ledger'}
api_output = get_audit_response(demo_audit, demo_chain, completion_event_id,
                               request_id='status-' + demo_verification_id)
assert api_output['data'] == confirmed_output
print(json.dumps(api_output, indent=2))
print('This is a real receipt from the local EVM; it is not a public-network receipt.')

### Cell 21 — Inspect the timeline and compare original vs changed bytes

In [ ]:
for row in demo_audit.history(demo_verification_id):
    envelope = json.loads(row['envelope_json'])
    print(row['sequence'], envelope['recorded_at'], envelope['request']['event_type'])
print('Original:', verify_document(demo_audit, completion_event_id,
                                  original_document_bytes, demo_chain))
print('Changed:', verify_document(demo_audit, completion_event_id,
                                 original_document_bytes + b'changed', demo_chain))

A matching file hash proves matching recorded bytes, not genuine identity. The next cell attacks a **copy** of the journal, rewrites the decision and recalculates all local hashes. The independent EVM commitments still expose the alteration.

### Cell 22 — Tampering demonstration — original journal is preserved

In [ ]:
with tempfile.TemporaryDirectory() as temporary:
    tampered_path = Path(temporary) / 'tampered.sqlite3'
    source_db = demo_audit.connect()
    target_db = sqlite3.connect(tampered_path)
    source_db.backup(target_db)
    source_db.close()
    target_db.close()
    tampered = AuditTrail(tampered_path)
    db = tampered.connect()
    db.execute('DROP TRIGGER no_update')  # Simulated malicious database administrator.
    previous = '0' * 64
    for row in tampered.events():
        envelope = json.loads(row['envelope_json'])
        if row['event_id'] == completion_event_id:
            envelope['request']['result']['riskResult']['decision'] = 'accept'
        envelope['previous_hash'] = previous
        encoded = canonical(envelope)
        current_hash = sha256(encoded.encode())
        db.execute('UPDATE events SET envelope_json=?, request_json=?, event_hash=? WHERE sequence=?',
                   (encoded, canonical(envelope['request']), current_hash, row['sequence']))
        previous = current_hash
    db.close()
    print('Rewritten local chain:', tampered.verify_local())
    try:
        verify_anchoring(tampered, demo_chain)
    except ValueError as error:
        print('EXPECTED: independent blockchain check detected tampering:', error)
    else:
        raise AssertionError('Tampering was not detected')
print('Original journal:', verify_anchoring(demo_audit, demo_chain))

### Cell 23 — Run all 39 package tests

In [ ]:
test_run = subprocess.run([sys.executable, '-m', 'unittest', 'discover',
                           '-s', 'tests', '-v'], capture_output=True, text=True)
print(test_run.stdout + test_run.stderr)
assert test_run.returncode == 0, 'Tests failed; inspect the report above.'
assert 'skipped=' not in test_run.stderr, 'Some test dependencies are unavailable.'

## Optional: use your team’s actual input
Default Run all skips these cells. Supply **two files** when enabled: a sample document and a JSON file containing the complete `AuditLogInput` object (`documentId`, `checkpointId`, `timestamp`, `riskResult`). Use synthetic/redacted examples while testing.

The notebook does not call an unknown backend URL or try to infer a missing risk output. Once Member 4’s result is in memory, the upload cell can be replaced with direct assignments to `team_document_bytes` and `team_audit_input`. The callable above is unchanged.

Your old Streamlit backend can retain original upload bytes using `document_file.getvalue()` before OpenCV preprocessing. Feed the final risk object into `riskResult`; do not convert old face/DOB branches into made-up scores.

### Cell 24 — Optional team-input upload

In [ ]:
USE_TEAM_UPLOAD = False  # Set True and run this cell manually when ready.
if USE_TEAM_UPLOAD:
    from google.colab import files
    print('Upload ONE sample document:')
    document_upload = files.upload()
    if len(document_upload) != 1:
        raise ValueError('Upload exactly one document')
    team_document_bytes = next(iter(document_upload.values()))
    print('Upload ONE AuditLogInput JSON file:')
    result_upload = files.upload()
    if len(result_upload) != 1:
        raise ValueError('Upload exactly one JSON file')
    team_audit_input = json.loads(next(iter(result_upload.values())).decode('utf-8'))
    from member5.contracts import validate_input
    print('Validation warnings:', validate_input(team_audit_input))
    # A new upload creates a new attempt. Rerun only the recording cell to retry.
    team_attempt_id = 'team-' + uuid.uuid4().hex
    print('Team input loaded; values are not printed.')
else:
    print('Skipped: synthetic demo remains the default.')

### Cell 25 — Optional team-input integration call

In [ ]:
LOG_TEAM_INPUT = False  # Enable after loading or assigning team input above.
if LOG_TEAM_INPUT:
    team_ack = submit_screening_to_audit(
        demo_audit, team_audit_input, team_document_bytes,
        event_id=team_attempt_id + ':completed', verification_id=team_attempt_id,
        actor_id='colab-demo-operator',  # Real backend uses authenticated context.
    )
    print(json.dumps(team_ack, indent=2))
    print('Recorded on the demo journal. Rerun the demo anchoring cell to anchor it.')
    print('Use get_audit_output(demo_audit, demo_chain, team_attempt_id + ":completed") afterward.')
else:
    print('Skipped: enable only when your input is ready.')

## Optional: connect an existing persistent EVM deployment
Your teammate must provision the network and deploy the included contract. Use a separate durable journal for that deployment. Do not mix its state with the disposable demo journal.

The following cell is read-only and disabled by default. It checks network ID and contract presence. Signing stays in the backend worker; no private key is embedded in this notebook. See `README.md` and `BACKEND_HANDOFF.md` for deployment, environment variables, the worker and retry rules. A Colab process is not a durable backend worker.

### Cell 26 — Optional read-only persistent-chain connector

In [ ]:
CONNECT_EXTERNAL_CHAIN = False
if CONNECT_EXTERNAL_CHAIN:
    from getpass import getpass
    rpc_url = getpass('Trusted RPC URL (hidden; may contain an API token): ')
    chain_id = int(input('Expected chain ID: '))
    contract_address = input('Deployed AuditAnchor address: ').strip()
    deployment_block = int(input('Deployment block number: '))
    confirmations = int(input('Required confirmation depth (>=1): '))
    external_reader = EVMAnchor(rpc_url, contract_address, chain_id,
        confirmations=confirmations, deployment_block=deployment_block)
    print('Read-only connection established. No signing key was requested.')
else:
    print('Skipped: no external network was contacted.')

## Export for backend integration
The source export uses repository-relative modules/blockchain/ paths and includes your modules, contract, tests, examples and handoff docs. Merge it at the GitHub repository root. Save this notebook separately in modules/blockchain/notebooks/ and the PDF in modules/blockchain/docs/. The delivered full upload ZIP already includes both; this runtime export contains source only. It deliberately excludes signing keys, uploaded documents and live journals. The export code lists the exact files, so unrelated runtime files cannot get swept into the ZIP.

For an operational journal backup use SQLite’s backup API and your team’s secure storage. A notebook/source ZIP does not retain the in-memory blockchain. Anchored evidence on a persistent network requires preserving both that deployment identity and the matching private journal.

### Cell 27 — Export the integration package

In [ ]:
EXPORT_FILES = ['.gitignore', 'BACKEND_HANDOFF.md', 'README.md', 'REPOSITORY_REVIEW.md', 'TEST_REPORT.md', 'contract_reference/barcodeScan.ts', 'contract_reference/blockchainAudit.ts', 'contract_reference/common.ts', 'contract_reference/documentValidation.ts', 'contract_reference/faceVerification.ts', 'contract_reference/index.ts', 'contract_reference/manifest.json', 'contract_reference/ocr.ts', 'contract_reference/riskScore.ts', 'contract_reference/tamperingDetection.ts', 'contract_reference/watermarkVerification.ts', 'contracts/AuditAnchor.json', 'contracts/AuditAnchor.sol', 'examples/audit_input.json', 'examples/demo.py', 'examples/evm_demo.py', 'examples/synthetic_document.txt', 'integration_types.ts', 'member5/__init__.py', 'member5/api_adapter.py', 'member5/audit_trail.py', 'member5/blockchain.py', 'member5/cli.py', 'member5/contracts.py', 'member5/integration.py', 'member5/worker.py', 'pyproject.toml', 'requirements-blockchain.txt', 'requirements-demo.txt', 'scripts/compile_contract.py', 'scripts/deploy_contract.py', 'scripts/verify_contracts.py', 'tests/contracts.typecheck.ts', 'tests/test_audit.py', 'tests/test_evm.py', 'tests/test_repo_contracts.py']
export_path = REPO_WORKSPACE / 'SIH_Member5_Colab_Source_Upload.zip'
with zipfile.ZipFile(export_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for relative in EXPORT_FILES:
        archive.write(PROJECT / relative, 'modules/blockchain/' + relative)
with zipfile.ZipFile(export_path) as archive:
    assert archive.testzip() is None
print('Created:', export_path)
from IPython.display import display, FileLink
display(FileLink(str(export_path)))
DOWNLOAD_ZIP = False  # Set True to start the browser download in Colab.
if DOWNLOAD_ZIP:
    from google.colab import files
    files.download(str(export_path))

## What to hand over
- **Member 4:** `member5/`, the exact input contract, and `BACKEND_HANDOFF.md`. They call the adapter after their final risk decision and persist retry IDs/jobs.
- **Member 6:** the confirmed receipt fields and pending-state behavior. A local ACK is not on-chain confirmation.
- **Your contribution:** persistent event journal, SHA-256 file/event commitments, Solidity contract, EVM connector, tamper checks and retry behavior.

**Limits:** blockchain records what was submitted; it does not prove the AI, document, timestamp or officer identity was truthful. SQLite is plaintext. Real authentication, authorization, encrypted storage, TLS, network governance and key custody belong to the integrated deployment.

**References:** [Colab FAQ](https://research.google.com/colaboratory/faq.html), [Web3.py API](https://web3py.readthedocs.io/en/v7.10.0/web3.eth.html), and [Solidity documentation](https://docs.soliditylang.org/en/latest/introduction-to-smart-contracts.html).

The notebook was prepared from the tested Member 5 package. The final notebook is verified with the default synthetic flow; interactive Google upload/download dialogs and a persistent external network require your Colab/team environment.
